In [ ]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

df = pd.read_csv('Supplementary data S6- input_combined.csv')
df


In [ ]:
# Hyperparameter search used to select the final RF configuration
# from sklearn.model_selection import GridSearchCV
#
# X = df.filter(like='PubchemFP')
# y = df['Activity'].map({'Inactive': 0, 'Active': 1})
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.2,
#     stratify=y,
#     random_state=42
# )
#
# param_grid = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [5, 10, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }
#
# grid_search = GridSearchCV(
#     RandomForestClassifier(random_state=42),
#     param_grid,
#     cv=5,
#     n_jobs=-1
# )
# grid_search.fit(X_train, y_train)
#
# best_params = grid_search.best_params_
# best_params


In [ ]:
X_rf = df.filter(like='PubchemFP')
y_rf = df['Activity'].map({'Inactive': 0, 'Active': 1})

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf,
    y_rf,
    test_size=0.2,
    stratify=y_rf,
    random_state=42
)

best_params_rf = {
    'max_depth': 30,
    'min_samples_leaf': 2,
    'min_samples_split': 10,
    'n_estimators': 100
}

best_rf_classifier = RandomForestClassifier(
    random_state=42,
    **best_params_rf
)

best_rf_classifier.fit(X_train_rf, y_train_rf)
y_pred_rf = best_rf_classifier.predict(X_test_rf)

accuracy_rf = accuracy_score(y_test_rf, y_pred_rf)
conf_matrix_rf = confusion_matrix(y_test_rf, y_pred_rf)

rf_base = 'Supplementary data S9- BCL2_RF_Abtin'
rf_name_rf = rf_base + '.pkl'
joblib.dump(best_rf_classifier, rf_name_rf)

print('RF Name:', rf_name_rf)
print('Accuracy:', accuracy_rf)
print('Confusion Matrix:\n', conf_matrix_rf)


In [ ]:
rf_model = 'Supplementary data S9- BCL2_RF_Abtin.pkl'
rf_base = 'Supplementary data S9- BCL2_RF_Abtin'
loaded_model = joblib.load(rf_model)

descriptor_csv = 'All_PDB_SwissSimilarity_descriptors.csv'
unseen_name = 'All_PDB_SwissSimilarity'
unseen_data = unseen_name + '.csv'

df_full = pd.read_csv(unseen_data)
compound_info = df_full[['SMILES', 'ID']]
descriptors = pd.read_csv(descriptor_csv)

unseen_combined = pd.concat([compound_info, descriptors], axis=1)
X_unseen = unseen_combined.filter(like='PubchemFP')

if X_unseen.shape[1] == 0:
    raise ValueError('No PubChem fingerprint columns were found in the descriptor table.')

y_pred = loaded_model.predict(X_unseen)

unseen_combined['Prediction'] = y_pred
final_result = pd.concat(
    [df_full, unseen_combined['Prediction']],
    axis=1
)

csv_pred = rf_base + '_pred_' + unseen_data
final_result.to_csv(csv_pred, index=False)

print(final_result.tail())
print("Count of 'Prediction' == 1:", (final_result['Prediction'] == 1).sum())
print('CSV Filename:', csv_pred)


In [ ]:
df_pred = pd.read_csv(csv_pred)

df_smiles = (
    df_pred.loc[df_pred['Prediction'] == 1, ['SMILES', 'ID']]
    .copy()
)

df_smiles['SMILES'] = df_smiles['SMILES'].str.strip(' "')

smiles_name = csv_pred.removesuffix('.csv') + '_active.smiles'
df_smiles.to_csv(
    smiles_name,
    sep='\t',
    index=False,
    header=False
)

df_smiles
